## Migrate Schema: `hls_glucosphere.cgm` → `hls_amer_catalog.glucosphere`

**Author:** May Merkletan  
**Date:** 2026-03-12  
**Purpose:** Bulk-migrate all **tables**, **volumes**, and **models** from `hls_glucosphere.cgm` into `hls_amer_catalog.glucosphere`, adding a `cgm_` prefix to each asset name.

### Task Plan
| Step | Task | Details |
| --- | --- | --- |
| 1 | **Setup & Widgets** | Define source/destination paths and table-name prefix via notebook widgets |
| 2 | **List Source Tables** | Dynamically query `SHOW TABLES` to collect all 34 table names |
| 3 | **Copy Tables** | Loop through each table and run `CREATE TABLE IF NOT EXISTS … AS SELECT *` |
| 4 | **Verify Tables** | Compare table counts and row counts between source and destination |
| 5 | **Copy Volumes** | Create destination volumes and `dbutils.fs.cp` files recursively (4 volumes) |
| 6 | **Re-register Models** | Load model artifacts from source MLflow runs and register in destination schema (2 models) |
| 7 | **Final Verification** | Confirm volumes, models, and tables all exist in destination |

### Asset Inventory
| Asset Type | Count | Items |
| --- | --- | --- |
| Tables | 34 | `baseline_for_validation_7d`, `diabetes_data`, `gold_patient_device_readings`, … |
| Volumes | 4 | `data`, `diabetes_who_pdf`, `landing_zone`, `optuna_studies` |
| Models | 2 | `cgm_xgb_30m` (v1), `cgm_xgb_15m` (v1) |

### Key Paths
* **Source schema:** `hls_glucosphere.cgm`
* **Destination schema:** `hls_amer_catalog.glucosphere`
* **Prefix:** `cgm_`

In [0]:
# -- Widgets ----------------------------------------------------------------
dbutils.widgets.text("source",      "hls_glucosphere.cgm",            "Source (catalog.schema)")
dbutils.widgets.text("destination", "hls_amer_catalog.glucosphere",   "Destination (catalog.schema)")
dbutils.widgets.text("prefix",      "cgm_",                           "Table Name Prefix")

# -- Read widget values -----------------------------------------------------
source:      str = dbutils.widgets.get("source")
destination: str = dbutils.widgets.get("destination")
prefix:      str = dbutils.widgets.get("prefix")

# -- Derived config ---------------------------------------------------------
src_catalog, src_schema   = source.split(".")
dest_catalog, dest_schema = destination.split(".")

print(f"Source      : {src_catalog}.{src_schema}")
print(f"Destination : {dest_catalog}.{dest_schema}")
print(f"Prefix      : {prefix}")

### Step 2 — List Source Tables

In [0]:
# Collect all table names from the source schema
tables_df = spark.sql(f"SHOW TABLES IN {src_catalog}.{src_schema}")
table_names: list[str] = [row.tableName for row in tables_df.collect()]

print(f"Found {len(table_names)} tables in {source}:")
for t in table_names:
    print(f"  {t}  →  {dest_catalog}.{dest_schema}.{prefix}{t}")

### Step 3 — Copy Tables with Prefix

In [0]:
from pyspark.sql.utils import AnalysisException

success: list[str] = []
failures: list[tuple[str, str]] = []

for tbl in table_names:
    src_fqn  = f"{src_catalog}.{src_schema}.{tbl}"
    dest_fqn = f"{dest_catalog}.{dest_schema}.{prefix}{tbl}"
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {dest_fqn} AS SELECT * FROM {src_fqn}")
        success.append(tbl)
        print(f"  ✓  {dest_fqn}")
    except AnalysisException as e:
        failures.append((tbl, str(e)))
        print(f"  ✗  {dest_fqn}  — {e}")

print(f"\n--- Done: {len(success)} succeeded, {len(failures)} failed ---")

### Step 4 — Verify Copied Tables

In [0]:
import pyspark.sql.functions as F

# List destination tables matching the prefix
dest_tables_df = (
    spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}")
    .filter(F.col("tableName").startswith(prefix))
)
dest_table_names: list[str] = [row.tableName for row in dest_tables_df.collect()]

print(f"Destination tables with prefix '{prefix}': {len(dest_table_names)} / {len(table_names)} expected")

# Row-count comparison
rows: list[dict] = []
for tbl in table_names:
    src_fqn  = f"{src_catalog}.{src_schema}.{tbl}"
    dest_fqn = f"{dest_catalog}.{dest_schema}.{prefix}{tbl}"
    src_cnt  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {src_fqn}").first().cnt
    try:
        dest_cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {dest_fqn}").first().cnt
    except Exception:
        dest_cnt = -1
    rows.append({"table": tbl, "source_rows": src_cnt, "dest_rows": dest_cnt, "match": src_cnt == dest_cnt})

verify_df = spark.createDataFrame(rows)
display(verify_df.orderBy("table"))

### Step 5 — Copy Volumes
Create matching managed volumes in the destination schema and recursively copy files with `dbutils.fs.cp`.

**Source volumes (4):** `data`, `diabetes_who_pdf`, `landing_zone`, `optuna_studies`

In [0]:
# List source volumes dynamically
vol_names: list[str] = [
    row.volume_name
    for row in spark.sql(f"SHOW VOLUMES IN {src_catalog}.{src_schema}").collect()
]
print(f"Found {len(vol_names)} volumes in {source}")

vol_success: list[str] = []
vol_failures: list[tuple[str, str]] = []

for vol in vol_names:
    dest_vol = f"{prefix}{vol}"
    src_path  = f"/Volumes/{src_catalog}/{src_schema}/{vol}"
    dest_path = f"/Volumes/{dest_catalog}/{dest_schema}/{dest_vol}"
    try:
        # Create managed volume in destination (idempotent)
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {dest_catalog}.{dest_schema}.{dest_vol}")
        # Recursively copy files
        dbutils.fs.cp(src_path, dest_path, recurse=True)
        vol_success.append(vol)
        print(f"  ✓  {dest_path}")
    except Exception as e:
        vol_failures.append((vol, str(e)))
        print(f"  ✗  {dest_path}  — {e}")

print(f"\n--- Volumes: {len(vol_success)} succeeded, {len(vol_failures)} failed ---")

### Step 6 — Re-register Models
Copy model versions from `hls_glucosphere.cgm` to `hls_amer_catalog.glucosphere` using the MLflow `copy_model_version` API.

**Source models (2):** `cgm_xgb_30m` (v1), `cgm_xgb_15m` (v1)

In [0]:
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Discover source models
all_models = client.search_registered_models(max_results=200)
src_models = [m for m in all_models if m.name.startswith(f"{src_catalog}.{src_schema}.")]
print(f"Found {len(src_models)} models in {source}")

model_success: list[str] = []
model_failures: list[tuple[str, str]] = []

for m in src_models:
    # Extract short name and apply prefix
    short_name = m.name.split(".")[-1]
    dest_model_name = f"{dest_catalog}.{dest_schema}.{prefix}{short_name}"

    try:
        # Ensure destination registered model exists
        try:
            client.create_registered_model(dest_model_name)
            print(f"  Created model: {dest_model_name}")
        except Exception:
            print(f"  Model already exists: {dest_model_name}")

        # Copy each version
        src_versions = client.search_model_versions(f"name='{m.name}'")
        for v in src_versions:
            copied = client.copy_model_version(
                src_model_uri=f"models:/{m.name}/{v.version}",
                dst_name=dest_model_name,
            )
            print(f"    ✓  v{v.version} → {dest_model_name} v{copied.version}")

        model_success.append(short_name)
    except Exception as e:
        model_failures.append((short_name, str(e)))
        print(f"    ✗  {dest_model_name}  — {e}")

print(f"\n--- Models: {len(model_success)} succeeded, {len(model_failures)} failed ---")

### Step 7 — Final Verification
Confirm all assets (tables, volumes, models) exist in the destination schema.

In [0]:
# --- Tables ---
dest_tbls = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
    if row.tableName.startswith(prefix)
]

# --- Volumes ---
dest_vols = [
    row.volume_name
    for row in spark.sql(f"SHOW VOLUMES IN {dest_catalog}.{dest_schema}").collect()
    if row.volume_name.startswith(prefix)
]

# --- Models ---
all_dest_models = client.search_registered_models(max_results=200)
dest_mdls = [m.name for m in all_dest_models if m.name.startswith(f"{dest_catalog}.{dest_schema}.{prefix}")]

# Summary
summary = [
    {"asset_type": "Tables",  "source_count": len(table_names), "dest_count": len(dest_tbls),  "status": "✓" if len(dest_tbls)  >= len(table_names) else "⚠"},
    {"asset_type": "Volumes", "source_count": len(vol_names),   "dest_count": len(dest_vols),  "status": "✓" if len(dest_vols)  >= len(vol_names)   else "⚠"},
    {"asset_type": "Models",  "source_count": len(src_models),  "dest_count": len(dest_mdls),  "status": "✓" if len(dest_mdls)  >= len(src_models)  else "⚠"},
]

summary_df = spark.createDataFrame(summary)
display(summary_df)